In [ ]:
# Ten plik nie wchodzi w sk
import pandas as pd
import asyncio
import os
from IPython.display import clear_output
from openai import AsyncOpenAI

client = AsyncOpenAI(api_key=os.getenv("OPENAI_API_KEY"))

df = pd.read_csv("../statics/datasets/reviews_New_York_short_cleaned_for_labelling.csv")

BATCH_SIZE = 50
MAX_CONCURRENT_BATCHES = 5
ASPECTS = ["safety", "cleanliness"]

SYSTEM_PROMPT = f"""You are a sentiment labeling assistant.
For each numbered review, classify sentiment for 'Safety' and 'Cleanliness'.
Use ONLY: Positive, Negative, Neutral, NotMentioned.
Reply with one line per review in the format:  NUMBER: {",".join(ASPECTS)}
Example output:
1: NotMentioned,Negative
2: Positive,Positive"""

def sanitize(text):
    if not isinstance(text, str) or not text.strip():
        return ""
    return text.replace("\x00", "").strip()

def build_batch_prompt(texts):
    lines = []
    for i, text in enumerate(texts, 1):
        clean = sanitize(str(text))
        lines.append(f"{i}: {clean}" if clean else f"{i}: [empty]")
    return "\n".join(lines)

def parse_batch_response(response_text, batch_size):
    results = [("Error", "Error")] * batch_size
    for line in response_text.strip().splitlines():
        line = line.strip()
        if not line or ":" not in line:
            continue
        try:
            num_str, labels = line.split(":", 1)
            idx = int(num_str.strip()) - 1
            parts = labels.strip().split(",")
            safety = parts[0].strip()
            cleanliness = parts[1].strip() if len(parts) > 1 else "None"
            if 0 <= idx < batch_size:
                results[idx] = (safety, cleanliness)
        except (ValueError, IndexError):
            continue
    return results

progress = {"done": 0, "total": 0, "errors": 0}

def update_progress():
    pct = progress["done"] / progress["total"] * 100 if progress["total"] else 0
    bar_len = 40
    filled = int(bar_len * progress["done"] // progress["total"]) if progress["total"] else 0
    bar = "█" * filled + "░" * (bar_len - filled)
    clear_output(wait=True)
    print(f"Progress: |{bar}| {pct:5.1f}%  ({progress['done']}/{progress['total']} rows, {progress['errors']} errors)")

async def label_batch(batch_idx, texts, semaphore):
    user_prompt = build_batch_prompt(texts)

    async with semaphore:
        try:
            response = await client.chat.completions.create(
                model="gpt-5.4-nano",
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt},
                ],
                temperature=0,
                # max_tokens=BATCH_SIZE * 15,
            )
        except Exception as e:
            print(f"[batch {batch_idx}] Error: {e}")
            progress["done"] += len(texts)
            progress["errors"] += len(texts)
            update_progress()
            return [("Error", "Error")] * len(texts)

    results = parse_batch_response(response.choices[0].message.content, len(texts))
    progress["done"] += len(texts)
    progress["errors"] += sum(1 for s, c in results if s == "Error")
    update_progress()
    return results

async def label_all(texts):
    semaphore = asyncio.Semaphore(MAX_CONCURRENT_BATCHES)
    batches = [texts[i : i + BATCH_SIZE] for i in range(0, len(texts), BATCH_SIZE)]

    progress["done"] = 0
    progress["total"] = len(texts)
    progress["errors"] = 0
    update_progress()

    tasks = [label_batch(i, batch, semaphore) for i, batch in enumerate(batches)]
    batch_results = await asyncio.gather(*tasks)

    flat = [pair for batch in batch_results for pair in batch]
    return flat

all_texts = df["text"].tolist()
results = await label_all(all_texts)

df[ASPECTS] = pd.DataFrame(results, columns=ASPECTS)

df.to_csv("labeled_reviews_output.csv", index=False)
print(f"Labeling complete ({len(df)} rows). Saved to labeled_reviews_output.csv")

Progress: |████████████████████████████████████████| 100.0%  (3000/3000 rows, 0 errors)
Labeling complete (3000 rows). Saved to labeled_reviews_output.csv
